# Trakπ — OTP vs MOTIS on Entur's RUT data

Two travel planners driven from the library and compared on the same Ruter (Oslo) timetable:
**OpenTripPlanner** (built locally) and **MOTIS** (`/workspace/motis-run`). Each has its own inline
`PlannerOrchestrator`, `TravelPlanner`, and `RequestLoader`, but they answer the **same neutral trip
requests** and report KPIs under the **same names**, so the results line up side by side.

OTP runs on port **8084**, MOTIS on **8085** (set in `motis-run/config.yml`), avoiding the
Docker-reserved 8081-8083, so both can serve at once.

## Data alignment — NeTEx for OTP, GTFS for MOTIS

Both engines use the **same Entur RUT (Oslo) timetable**, but in each engine's native format:

- **OTP** builds from Entur's **NeTEx** (`oslo-shared/netex`, the Ruter codespace + `_stops.xml`) and
  `oslo.osm.pbf`.
- **MOTIS** imports Entur's **GTFS** for the same codespace (`rb_rut-aggregated-gtfs.zip`).

Why not the same NeTEx for both? MOTIS imports Entur's NeTEx stops and journeys but doesn't resolve
the Nordic-profile service calendar (`AvailabilityCondition` + `OperatingDay`, with no per-journey
`DayTypeRef`), so every trip ends up with zero operating days and MOTIS returns no transit. Entur's
GTFS carries an ordinary `calendar.txt`, which MOTIS handles natively. So this is the same source,
region, and period — GTFS is a downstream subset of the NeTEx, so expect minor differences beyond
the engines themselves.

## 1. Dependencies

In [1]:
@file:DependsOn("/workspace/trakpi/core/common/target/common-1.0.0-SNAPSHOT.jar")
@file:DependsOn("/workspace/trakpi/core/tester/target/tester-1.0.0-SNAPSHOT.jar")
@file:DependsOn("/workspace/trakpi/core/orchestrator/target/orchestrator-1.0.0-SNAPSHOT.jar")

In [2]:
import java.net.URI
import java.net.URLEncoder
import java.net.http.HttpClient
import java.net.http.HttpRequest
import java.net.http.HttpResponse.BodyHandlers
import java.io.File
import java.nio.file.Files
import java.nio.file.Path
import java.nio.file.StandardCopyOption
import java.time.Duration
import java.time.Instant
import java.util.concurrent.TimeUnit
import java.util.zip.ZipInputStream
import kotlinx.serialization.json.*

import org.opentripplanner.trakpi.common.PlannerVersion
import org.opentripplanner.trakpi.common.TestsetVersion
import org.opentripplanner.trakpi.orchestrator.PlannerOrchestrator
import org.opentripplanner.trakpi.orchestrator.PlannerStatus
import org.opentripplanner.trakpi.tester.Tester
import org.opentripplanner.trakpi.tester.spi.*
import org.opentripplanner.trakpi.tester.spi.kpi.*

## 2. Neutral requests, one loader per planner

A request is a planner-neutral from/to spec (JSON). Each planner's `RequestLoader` turns that same
spec into its own native request — a Transmodel GraphQL query for OTP, a `/api/v1/plan` from/to pair
for MOTIS — so both run the same trips.

In [3]:
data class TripSpec(val fromLat: Double, val fromLon: Double, val toLat: Double, val toLon: Double)

fun parseSpec(body: String): TripSpec {
    val o = Json.parseToJsonElement(body).jsonObject
    fun d(k: String) = o[k]!!.jsonPrimitive.double
    return TripSpec(d("fromLat"), d("fromLon"), d("toLat"), d("toLon"))
}

fun spec(fromLat: Double, fromLon: Double, toLat: Double, toLon: Double) =
    buildJsonObject {
        put("fromLat", fromLat); put("fromLon", fromLon); put("toLat", toLat); put("toLon", toLon)
    }.toString()

// id -> neutral spec. Intra-Oslo trips within the Ruter dataset.
val requests = mapOf(
    "oslos-nationalth" to spec(59.911, 10.750, 59.914, 10.733),
    "oslos-majorstuen" to spec(59.911, 10.750, 59.929, 10.714),
    "oslos-blindern"   to spec(59.911, 10.750, 59.939, 10.721),
)

class InlineRequestFileLoader(private val bodies: Map<String, String>) : RequestFileLoader {
    override fun loadAll(testsetVersion: TestsetVersion): List<RequestFile> =
        bodies.map { (id, body) -> RequestFile(id, body) }
}

## 3. The two planners

`OtpPlanner` POSTs a GraphQL `trip` query; `MotisPlanner` GETs `/api/v1/plan`. Both return the opaque
raw body plus the generic success/method facts trakpi needs.

In [5]:
val CLIENT_NAME = "entur-trakpi-notebook"

// ---- OTP ----
data class GraphQlRequest(val query: String) : TravelPlannerRequest

class OtpRequestLoader : RequestLoader<GraphQlRequest> {
    override fun load(file: RequestFile): GraphQlRequest {
        val s = parseSpec(file.body)
        return GraphQlRequest(
            "{ trip(from: {coordinates: {latitude: ${s.fromLat}, longitude: ${s.fromLon}}}, " +
            "to: {coordinates: {latitude: ${s.toLat}, longitude: ${s.toLon}}}) " +
            "{ tripPatterns { duration legs { mode aimedStartTime aimedEndTime " +
            "fromPlace { name } toPlace { name } line { publicCode } } } debugOutput { totalTime } } }"
        )
    }
}

class OtpPlanner(private val endpoint: String, private val clientName: String) : TravelPlanner<GraphQlRequest> {
    private val http = HttpClient.newBuilder()
        .version(HttpClient.Version.HTTP_1_1).connectTimeout(Duration.ofSeconds(10)).build()
    override fun execute(request: GraphQlRequest): TravelPlannerResponse {
        val body = buildJsonObject { put("query", JsonPrimitive(request.query)) }.toString()
        val res = http.send(
            HttpRequest.newBuilder().uri(URI.create(endpoint))
                .header("Content-Type", "application/json").header("ET-Client-Name", clientName)
                .timeout(Duration.ofSeconds(30)).POST(HttpRequest.BodyPublishers.ofString(body)).build(),
            BodyHandlers.ofString())
        val json = runCatching { Json.parseToJsonElement(res.body()).jsonObject }.getOrNull()
        val data = json?.get("data") as? JsonObject
        val errors = json?.get("errors") as? JsonArray
        return TravelPlannerResponse(res.body(), res.statusCode() / 100 == 2 && data != null && errors.isNullOrEmpty(),
            data?.keys?.firstOrNull() ?: "unknown", mapOf("http_status_code" to res.statusCode().toString()))
    }
}

// ---- MOTIS ----
data class MotisRequest(val fromPlace: String, val toPlace: String) : TravelPlannerRequest

class MotisRequestLoader : RequestLoader<MotisRequest> {
    override fun load(file: RequestFile): MotisRequest {
        val s = parseSpec(file.body)
        return MotisRequest("${s.fromLat},${s.fromLon}", "${s.toLat},${s.toLon}")
    }
}

// radiusMeters: this MOTIS config has osr_footpath=false, so it won't street-route a bare coordinate
// to nearby stops. `radius` makes it use every stop within that many metres as an access point
// (crow-fly), which is what lets a coordinate query reach transit at all.
class MotisPlanner(private val endpoint: String, private val radiusMeters: Int = 1000) : TravelPlanner<MotisRequest> {
    private val http = HttpClient.newBuilder().connectTimeout(Duration.ofSeconds(10)).build()
    private fun enc(s: String) = URLEncoder.encode(s, "UTF-8")
    override fun execute(request: MotisRequest): TravelPlannerResponse {
        val url = "$endpoint/api/v1/plan?fromPlace=${enc(request.fromPlace)}" +
            "&toPlace=${enc(request.toPlace)}&radius=$radiusMeters"
        val res = http.send(
            HttpRequest.newBuilder().uri(URI.create(url)).timeout(Duration.ofSeconds(30)).GET().build(),
            BodyHandlers.ofString())
        val itineraries = runCatching { Json.parseToJsonElement(res.body()).jsonObject["itineraries"] as? JsonArray }.getOrNull()
        return TravelPlannerResponse(res.body(), res.statusCode() / 100 == 2 && itineraries != null,
            "plan", mapOf("http_status_code" to res.statusCode().toString()))
    }
}

## 4. KPIs — same names, per-planner parsing

Each planner emits its numbers from its own response shape, but under **identical KPI names**
(`itineraryCount`, `minDurationMin`, `success`) so they compare directly. Durations are seconds in
both formats.

In [6]:
fun TravelPlannerResponse.otpTrip(): JsonObject? =
    runCatching { Json.parseToJsonElement(raw).jsonObject["data"]?.jsonObject?.get("trip")?.jsonObject }.getOrNull()
fun TravelPlannerResponse.motisItineraries(): JsonArray? =
    runCatching { Json.parseToJsonElement(raw).jsonObject["itineraries"] as? JsonArray }.getOrNull()
fun durationsSec(patterns: JsonArray): List<Long> =
    patterns.mapNotNull { it.jsonObject["duration"]?.jsonPrimitive?.longOrNull }

// ---- OTP ----
class OtpItineraryCount : KPICalculator {
    override fun calculate(r: TravelPlannerResponse): Kpi? {
        val tp = r.otpTrip()?.get("tripPatterns") as? JsonArray ?: return null
        return Kpi("itineraryCount", tp.size.toDouble())
    }
}
class OtpMinDuration : KPICalculator {
    override fun calculate(r: TravelPlannerResponse): Kpi? {
        val tp = r.otpTrip()?.get("tripPatterns") as? JsonArray ?: return null
        val min = durationsSec(tp).minOrNull() ?: return null
        return Kpi("minDurationMin", min / 60.0)
    }
}

// ---- MOTIS ----
class MotisItineraryCount : KPICalculator {
    override fun calculate(r: TravelPlannerResponse): Kpi? {
        val its = r.motisItineraries() ?: return null
        return Kpi("itineraryCount", its.size.toDouble())
    }
}
class MotisMinDuration : KPICalculator {
    override fun calculate(r: TravelPlannerResponse): Kpi? {
        val its = r.motisItineraries() ?: return null
        val min = durationsSec(its).minOrNull() ?: return null
        return Kpi("minDurationMin", min / 60.0)
    }
}

// ---- shared ----
class SuccessKpi : KPICalculator {
    override fun calculate(r: TravelPlannerResponse) = Kpi("success", if (r.success) 1.0 else 0.0)
}

// ---- comparative: MOTIS (subject) vs OTP (reference), same request ----
fun itinCount(r: TravelPlannerResponse): Int? =
    r.motisItineraries()?.size ?: (r.otpTrip()?.get("tripPatterns") as? JsonArray)?.size
fun minDurationSec(r: TravelPlannerResponse): Long? =
    (r.motisItineraries() ?: r.otpTrip()?.get("tripPatterns") as? JsonArray)?.let { durationsSec(it).minOrNull() }

class ItineraryCountDeltaVsOtp : ComparativeKPICalculator {
    override fun calculate(subject: TravelPlannerResponse, reference: TravelPlannerResponse): Kpi? {
        val s = itinCount(subject) ?: return null; val r = itinCount(reference) ?: return null
        return Kpi("itinCountDeltaVsOtp", (s - r).toDouble())
    }
}
class MinDurationDeltaVsOtp : ComparativeKPICalculator {
    override fun calculate(subject: TravelPlannerResponse, reference: TravelPlannerResponse): Kpi? {
        val s = minDurationSec(subject) ?: return null; val r = minDurationSec(reference) ?: return null
        return Kpi("minDurationDeltaMinVsOtp", (s - r) / 60.0)
    }
}

## 4b. Semantic itinerary fingerprints (fuzzy, transit legs)

Like trakpi's reference-OTP comparison, each itinerary is fingerprinted by its **ordered legs** — but
loosened for the cross-engine case. Two itineraries count as the same journey when their **transit
legs** match on canonical mode, line code, and **minute-precision** scheduled departure/arrival:

- **Walk legs are ignored** — access/egress/transfer walking differs between engines and shouldn't
  break a match.
- **Times are minute-precision** (`HH:MM`, Oslo-local) — sub-minute differences don't matter.
- Line code bridges the formats (NeTEx `line.publicCode` = GTFS `routeShortName`, e.g. `BUS 30`).

So a fingerprint is the sequence of `MODE line dep-arr` for the transit legs only; a walk-only
itinerary fingerprints to empty and is excluded from the match.

In [7]:
fun canonMode(m: String): String = when (m.uppercase()) {
    "FOOT", "WALK" -> "WALK"
    "SUBWAY", "METRO" -> "METRO"
    "RAIL", "REGIONAL_RAIL", "REGIONAL_FAST_RAIL", "LONG_DISTANCE", "HIGHSPEED_RAIL", "NIGHT_RAIL" -> "RAIL"
    "WATER", "FERRY" -> "FERRY"
    "AIR", "AIRPLANE" -> "AIR"
    else -> m.uppercase()
}

// ISO offset or Z time -> Oslo-local HH:MM (both engines' scheduled times compare on the same clock).
fun hhmm(iso: String?): String = if (iso == null) "--:--" else runCatching {
    java.time.OffsetDateTime.parse(iso).atZoneSameInstant(java.time.ZoneId.of("Europe/Oslo")).toLocalTime().toString().take(5)
}.getOrDefault("--:--")

data class SemLeg(val mode: String, val line: String, val from: String, val to: String, val dep: String, val arr: String) {
    val pretty: String get() = if (mode == "WALK") "walk $dep-$arr" else "$mode $line ($from $dep -> $to $arr)"
}
data class SemItin(val durationSec: Long, val legs: List<SemLeg>) {
    // Fuzzy cross-engine key: transit legs only (walk ignored), each keyed by mode, line, and
    // minute-precision scheduled dep/arr. Two itineraries are the same journey iff these match.
    val transitFingerprint: List<String> get() =
        legs.filter { it.mode != "WALK" }.map { "${it.mode} ${it.line} ${it.dep}-${it.arr}" }
    fun render(): String {
        val head = "(${durationSec / 60}m)"
        return if (legs.isEmpty()) "$head (empty)" else "$head " + legs.joinToString("  ->  ") { it.pretty }
    }
}

private fun legsOf(o: JsonObject) = (o["legs"] as? JsonArray).orEmpty().map { it.jsonObject }
private fun str(o: JsonObject, k: String) = o[k]?.jsonPrimitive?.contentOrNull
private fun lng(o: JsonObject, k: String) = o[k]?.jsonPrimitive?.longOrNull ?: 0L
private fun name(o: JsonObject, k: String) = (o[k] as? JsonObject)?.get("name")?.jsonPrimitive?.contentOrNull ?: ""

fun otpItineraries(r: TravelPlannerResponse): List<SemItin> =
    ((r.otpTrip()?.get("tripPatterns")) as? JsonArray).orEmpty().map { it.jsonObject }.map { p ->
        SemItin(lng(p, "duration"), legsOf(p).map { l -> SemLeg(
            canonMode(str(l, "mode") ?: "?"),
            (l["line"] as? JsonObject)?.get("publicCode")?.jsonPrimitive?.contentOrNull ?: "",
            name(l, "fromPlace"), name(l, "toPlace"),
            hhmm(str(l, "aimedStartTime")), hhmm(str(l, "aimedEndTime"))) })
    }

fun motisItineraries2(r: TravelPlannerResponse): List<SemItin> =
    (r.motisItineraries()).orEmpty().map { it.jsonObject }.map { p ->
        SemItin(lng(p, "duration"), legsOf(p).map { l -> SemLeg(
            canonMode(str(l, "mode") ?: "?"),
            str(l, "routeShortName") ?: "",
            name(l, "from"), name(l, "to"),
            hhmm(str(l, "scheduledStartTime")), hhmm(str(l, "scheduledEndTime"))) })
    }

// The set of transit fingerprints in a response; walk-only itineraries (empty fingerprint) are dropped.
fun transitFingerprints(itins: List<SemItin>): Set<List<String>> =
    itins.map { it.transitFingerprint }.filter { it.isNotEmpty() }.toSet()

// 1.0 when MOTIS returns the same set of transit itineraries as OTP (matched on line + minute-precision
// dep/arr, walk legs ignored), else 0.0 — the fuzzy cross-engine analog of the reference project's
// ItinerariesMatchReferenceKPICalculator.
class ItinerariesMatchOtp : ComparativeKPICalculator {
    override fun calculate(subject: TravelPlannerResponse, reference: TravelPlannerResponse): Kpi? {
        reference.otpTrip() ?: return null
        val motis = transitFingerprints(motisItineraries2(subject))
        val otp = transitFingerprints(otpItineraries(reference))
        return Kpi("itinerariesMatchOtp", if (motis == otp) 1.0 else 0.0)
    }
}

## 5. Results writer — tagged by planner

`RunMetadata.application` names the planner, so a writer can tag each result with which engine
produced it.

In [8]:
data class ResultRow(val planner: String, val result: TestCaseResult)
val collected = mutableListOf<ResultRow>()

class TaggingWriter : ResultsWriter {
    override fun store(run: RunMetadata, result: TestCaseResult) { collected += ResultRow(run.application, result) }
}

// Feed one run's responses back in as the reference the comparative KPIs read (matched by requestId).
fun TestCaseResult.asResponse() = TravelPlannerResponse(rawResponse, success, method, attributes)
class InMemoryResultsReader(private val byId: Map<String, TravelPlannerResponse>) : ResultsReader {
    override fun responses(version: PlannerVersion, testsetVersion: TestsetVersion) = byId
    override fun response(version: PlannerVersion, testsetVersion: TestsetVersion, requestId: String) = byId[requestId]
}

## 6. The two orchestrators

Both implement the `PlannerOrchestrator` SPI with an eager `start` and a pollable `status`. OTP runs
`java -jar <otp> --load <graph>`; MOTIS runs `./motis server` from its run directory.

In [9]:
// --- data preparation helpers (pure Kotlin; no shell) ---
val NETEX_BASE_URL = "https://storage.googleapis.com/marduk-production/outbound/netex"
val GTFS_BASE_URL = "https://storage.googleapis.com/marduk-production/outbound/gtfs"
val NORWAY_NETEX: Path = Path.of("/workspace/otp-data/otp/norway/netex")
val RUT_VIEW: Path = Path.of("/workspace/motis-run/input/rut-netex")

// Launch a planner build/import as a subprocess; output appended to prepare.log.
fun run(vararg cmd: String, dir: File? = null) {
    val p = ProcessBuilder(*cmd).directory(dir).redirectErrorStream(true)
        .redirectOutput(ProcessBuilder.Redirect.appendTo(File("prepare.log"))).start()
    check(p.waitFor() == 0) { "command failed: ${cmd.joinToString(" ")} - see prepare.log" }
}

fun httpDownload(url: String, dest: Path) {
    val http = HttpClient.newBuilder().followRedirects(HttpClient.Redirect.NORMAL).connectTimeout(Duration.ofSeconds(30)).build()
    val res = http.send(HttpRequest.newBuilder(URI.create(url)).GET().build(), BodyHandlers.ofInputStream())
    check(res.statusCode() / 100 == 2) { "download failed ($url): HTTP ${res.statusCode()}" }
    res.body().use { Files.copy(it, dest, StandardCopyOption.REPLACE_EXISTING) }
}

// Extract every entry flat (by file name) into destDir.
fun unzipFlat(zip: Path, destDir: Path) {
    ZipInputStream(Files.newInputStream(zip)).use { zis ->
        var e = zis.nextEntry
        while (e != null) {
            if (!e.isDirectory) Files.copy(zis, destDir.resolve(Path.of(e.name).fileName), StandardCopyOption.REPLACE_EXISTING)
            e = zis.nextEntry
        }
    }
}

fun Path.entries(): List<Path> = Files.newDirectoryStream(this).use { it.toList() }

// Download one codespace fresh from Entur into norway/netex (a single codespace, not full Norway).
fun downloadNetex(codespace: String) {
    val cs = codespace.lowercase(); val CS = codespace.uppercase()
    Files.createDirectories(NORWAY_NETEX)
    val zip = NORWAY_NETEX.resolve("_$cs.zip")
    println("[prepare] download $CS netex from Entur")
    httpDownload("$NETEX_BASE_URL/rb_$cs-aggregated-netex.zip", zip)
    NORWAY_NETEX.entries().filter { val n = it.fileName.toString(); n.startsWith("${CS}_") || n.startsWith("_${CS}_") }.forEach { Files.deleteIfExists(it) }
    unzipFlat(zip, NORWAY_NETEX)
    Files.deleteIfExists(zip)
}

// Rebuild the RUT-only view (line + shared files + the stable stops file) both engines read.
fun refreshRutView() {
    Files.createDirectories(RUT_VIEW)
    val keep = { n: String -> n.startsWith("RUT_") || n.startsWith("_RUT_") || n == "_stops.xml" }
    RUT_VIEW.entries().filter { keep(it.fileName.toString()) }.forEach { Files.deleteIfExists(it) }
    NORWAY_NETEX.entries().filter { keep(it.fileName.toString()) }.forEach { Files.createSymbolicLink(RUT_VIEW.resolve(it.fileName), it) }
}

// ---- readiness probes ----
fun otpResponds(endpoint: String, clientName: String): Boolean = try {
    val http = HttpClient.newBuilder().version(HttpClient.Version.HTTP_1_1).connectTimeout(Duration.ofSeconds(5)).build()
    val res = http.send(HttpRequest.newBuilder().uri(URI.create(endpoint))
        .header("Content-Type", "application/json").header("ET-Client-Name", clientName).timeout(Duration.ofSeconds(10))
        .POST(HttpRequest.BodyPublishers.ofString("""{"query":"{ serverInfo { version } }"}""")).build(), BodyHandlers.ofString())
    val data = Json.parseToJsonElement(res.body()).jsonObject["data"] as? JsonObject
    res.statusCode() / 100 == 2 && data?.get("serverInfo") != null
} catch (e: Exception) { false }

fun motisResponds(endpoint: String): Boolean = try {
    val http = HttpClient.newBuilder().connectTimeout(Duration.ofSeconds(5)).build()
    // Probe the plan endpoint with a fixed Oslo pair; ready once it answers with an itineraries array.
    val res = http.send(HttpRequest.newBuilder()
        .uri(URI.create("$endpoint/api/v1/plan?fromPlace=59.911%2C10.750&toPlace=59.914%2C10.733&radius=1000"))
        .timeout(Duration.ofSeconds(8)).GET().build(), BodyHandlers.ofString())
    res.statusCode() / 100 == 2 && Json.parseToJsonElement(res.body()).jsonObject["itineraries"] != null
} catch (e: Exception) { false }

// ---- a small base to share the process lifecycle + poll loop ----
abstract class ProcessOrchestrator : PlannerOrchestrator {
    protected var process: Process? = null
    abstract val logFile: String
    protected abstract fun command(): List<String>
    protected open fun workingDir(): File? = null
    protected abstract fun responds(): Boolean
    /** Engine-specific preparation after the shared NeTEx download: build (OTP) or import (MOTIS). */
    protected abstract fun prepareEngine()

    // prepare(): each engine fetches and preprocesses its own data — see prepareEngine.
    override fun prepare(version: PlannerVersion, args: String?) = prepareEngine()

    override fun start(version: PlannerVersion, args: String?) {
        // Reuse an instance already serving on this port (e.g. one left running by an earlier cell
        // run) instead of launching a clashing one. Stop/kill it first to force a fresh start.
        if (responds()) { println("[orchestrator] endpoint already serving - reusing it"); return }
        check(process == null) { "already launched - stop() first" }
        println("[orchestrator] ${command().joinToString(" ")}  (logs -> $logFile)")
        process = ProcessBuilder(command()).directory(workingDir()).redirectErrorStream(true)
            .redirectOutput(ProcessBuilder.Redirect.appendTo(File(logFile))).start()
    }
    override fun stop(version: PlannerVersion) {
        process?.let { it.destroy(); if (!it.waitFor(20, TimeUnit.SECONDS)) it.destroyForcibly() }
        process = null
    }
    override fun status(version: PlannerVersion): PlannerStatus {
        val p = process
        return when {
            responds() -> PlannerStatus.READY   // launched or reused
            p != null && !p.isAlive -> PlannerStatus.FAILED
            p != null -> PlannerStatus.STARTING
            else -> PlannerStatus.NOT_RUNNING
        }
    }
    fun awaitReady(version: PlannerVersion, attempts: Int = 12, interval: Duration = Duration.ofSeconds(5)): PlannerStatus {
        repeat(attempts) {
            when (val s = status(version)) {
                PlannerStatus.READY, PlannerStatus.FAILED, PlannerStatus.NOT_RUNNING -> return s
                PlannerStatus.STARTING -> Thread.sleep(interval.toMillis())
            }
        }
        return status(version)
    }
}

class LocalOtpOrchestrator(val otpJar: String, val graphDir: String, val port: Int = 8084,
    val heap: String = "-Xmx8g", override val logFile: String = "otp-server.log") : ProcessOrchestrator() {
    val endpoint = "http://localhost:$port/otp/transmodel/v3"
    override fun command() = listOf("java", heap, "-jar", otpJar, "--load", graphDir, "--port", "$port")
    override fun responds() = otpResponds(endpoint, CLIENT_NAME)
    override fun prepareEngine() {
        downloadNetex("RUT"); refreshRutView()
        println("[prepare] building OTP graph from NeTEx (slow) -> $graphDir/graph.obj")
        run("java", heap, "-jar", otpJar, "--build", "--save", graphDir)
    }
}

class MotisOrchestrator(val motisRunDir: String, val port: Int = 8085,
    // The MOTIS binary must match the OS running this notebook. The one shipped in motis-run is
    // linux-arm64; on macOS download the macos-arm64 build (see the note above) and pass its path here.
    val binary: String = "$motisRunDir/motis",
    override val logFile: String = "motis-server.log") : ProcessOrchestrator() {
    val endpoint = "http://localhost:$port"
    // Absolute binary path, but cwd = motisRunDir so config.yml's relative paths (osm, ui, datasets) resolve.
    override fun command() = listOf(binary, "server")
    override fun workingDir() = File(motisRunDir)
    override fun responds() = motisResponds(endpoint)
    override fun prepareEngine() {
        // MOTIS doesn't resolve Entur's NeTEx service calendar (trips end up with no operating days),
        // so feed it Entur's GTFS for the same codespace. config.yml's dataset points at rut-gtfs.zip.
        println("[prepare] download RUT GTFS from Entur")
        httpDownload("$GTFS_BASE_URL/rb_rut-aggregated-gtfs.zip", File("$motisRunDir/input/rut-gtfs.zip").toPath())
        println("[prepare] importing MOTIS (GTFS) -> $motisRunDir/data")
        run(binary, "import", dir = File(motisRunDir))
    }
}

## 7. Prepare both — OTP builds NeTEx, MOTIS imports GTFS

`prepare()` fetches each engine's data and preprocesses it: OTP downloads Entur's RUT **NeTEx** and
builds `graph.obj`; MOTIS downloads Entur's RUT **GTFS** (`rb_rut-aggregated-gtfs.zip`) and imports
it. Both come from Entur's marduk bucket, reachable from your Mac. Slow and heap-hungry; run once,
then comment out. Watch `prepare.log`.

**MOTIS binary must match your OS.** The one shipped in `motis-run/` is linux-arm64 and can't run
on macOS. An Apple-silicon build is already in place at `motis-run/motis-macos` (Mach-O arm64), and
the orchestrator below points at it. To re-fetch it, or for a different arch:

```bash
# TARGET: macos-arm64 (Apple silicon) | macos-x86_64 (Intel Mac) | linux-arm64 | linux-amd64
curl -L -o /tmp/motis.tar.bz2 \
  https://github.com/motis-project/motis/releases/download/v2.11.3/motis-macos-arm64.tar.bz2
tar xf /tmp/motis.tar.bz2 -C /tmp && cp /tmp/motis /workspace/motis-run/motis-macos
```

In [10]:
val otp = LocalOtpOrchestrator("/workspace/otp/otp-shaded/target/otp-shaded-2.10.0-SNAPSHOT.jar", "/workspace/otp-data/otp/oslo-shared")
// macOS arm64 MOTIS binary — the linux-arm64 one in motis-run/ can't run on macOS.
// On Linux, drop the `binary` arg to use motis-run/motis.
val motis = MotisOrchestrator("/workspace/motis-run", binary = "/workspace/motis-run/motis-macos")
val otpV = PlannerVersion("otp-local")
val motisV = PlannerVersion("motis-2.11.3")

// Run once (or for fresh data), then comment out:
otp.prepare(otpV, null)
motis.prepare(motisV, null)

[prepare] download RUT netex from Entur
[prepare] building OTP graph from NeTEx (slow) -> /workspace/otp-data/otp/oslo-shared/graph.obj
[prepare] download RUT GTFS from Entur
[prepare] importing MOTIS (GTFS) -> /workspace/motis-run/data


## 8. Start both planners, poll each until ready

Start both (each returns at once), then re-run the poll cell until both report `READY`. `FAILED`
means the process died — read `otp-server.log` / `motis-server.log`.

In [11]:
otp.start(otpV, null)
motis.start(motisV, null)

[orchestrator] endpoint already serving - reusing it
[orchestrator] /workspace/motis-run/motis-macos server  (logs -> motis-server.log)


In [12]:
"otp=${otp.awaitReady(otpV)}  motis=${motis.awaitReady(motisV)}"   // re-run until both READY

otp=READY  motis=READY

## 9. Run the same requests against both

OTP runs first and its responses become the **reference**. MOTIS then runs with the same requests
plus `ComparativeKPICalculator`s that score each MOTIS response against OTP's for that request —
trakpi's built-in comparative mechanism, fed by an in-memory `ResultsReader`.

In [13]:
check(otp.status(otpV) == PlannerStatus.READY && motis.status(motisV) == PlannerStatus.READY) {
    "both planners must be READY - re-run the poll cell"
}
collected.clear()

// 1. OTP — the reference run.
Tester(
    run = RunMetadata.create(version = otpV, application = "otp", startedAt = Instant.now(),
        testsetVersion = TestsetVersion("oslo-shared"), origin = "notebook"),
    requestFileLoader = InlineRequestFileLoader(requests),
    requestLoader = OtpRequestLoader(),
    travelPlanner = OtpPlanner(otp.endpoint, CLIENT_NAME),
    kpiCalculators = listOf(OtpItineraryCount(), OtpMinDuration(), SuccessKpi()),
    resultsWriter = TaggingWriter(),
).run()

// Feed OTP's responses back as the reference for the comparative KPIs.
val otpReader = InMemoryResultsReader(
    collected.filter { it.planner == "otp" }.associate { it.result.requestId to it.result.asResponse() })

// 2. MOTIS — with comparative KPIs against the OTP reference.
Tester(
    run = RunMetadata.create(version = motisV, application = "motis", startedAt = Instant.now(),
        testsetVersion = TestsetVersion("oslo-shared"), origin = "notebook", referenceVersion = otpV),
    requestFileLoader = InlineRequestFileLoader(requests),
    requestLoader = MotisRequestLoader(),
    travelPlanner = MotisPlanner(motis.endpoint),
    kpiCalculators = listOf(MotisItineraryCount(), MotisMinDuration(), SuccessKpi()),
    comparativeKpiCalculators = listOf(ItineraryCountDeltaVsOtp(), MinDurationDeltaVsOtp(), ItinerariesMatchOtp()),
    resultsReader = otpReader,
    referenceVersion = otpV,
    resultsWriter = TaggingWriter(),
).run()

"Collected ${collected.size} rows (${collected.map { it.planner }.distinct()})"

100% complete. 3/3 requests processed.
100% complete. 3/3 requests processed.


Collected 6 rows ([otp, motis])

## 10. Compare

Per-planner KPIs side by side, then the comparative KPIs (MOTIS minus OTP) that MOTIS computed
against the OTP reference. A `Δ` of 0 itineraries / small minutes means the engines broadly agree.

In [14]:
val requestIds = collected.map { it.result.requestId }.distinct()
fun fmt(v: Double?) = when {
    v == null -> "-"
    v == v.toLong().toDouble() -> v.toLong().toString()
    else -> "%.1f".format(v)
}
fun kpi(planner: String, req: String, name: String): Double? =
    collected.firstOrNull { it.planner == planner && it.result.requestId == req }
        ?.result?.kpis?.firstOrNull { it.name == name }?.value

val cols = listOf(
    "otp itins"      to { req: String -> kpi("otp", req, "itineraryCount") },
    "otp minDur"     to { req: String -> kpi("otp", req, "minDurationMin") },
    "motis itins"    to { req: String -> kpi("motis", req, "itineraryCount") },
    "motis minDur"   to { req: String -> kpi("motis", req, "minDurationMin") },
    "\u0394 itins"     to { req: String -> kpi("motis", req, "itinCountDeltaVsOtp") },
    "\u0394 minDur"    to { req: String -> kpi("motis", req, "minDurationDeltaMinVsOtp") },
)
val head = "<tr><th>request</th>" + cols.joinToString("") { "<th>${it.first}</th>" } + "</tr>"
val body = requestIds.joinToString("") { req ->
    "<tr><td>$req</td>" + cols.joinToString("") { "<td>${fmt(it.second(req))}</td>" } + "</tr>"
}
HTML("<table>$head$body</table>")

request,otp itins,otp minDur,motis itins,motis minDur,Δ itins,Δ minDur
oslos-nationalth,23,9.0,7,5,-16,-4.0
oslos-majorstuen,44,19.5,7,8,-37,-11.5
oslos-blindern,17,22.1,5,12,-12,-10.1


## 11. All itineraries, in full

Every itinerary each engine returned for each request, rendered leg by leg (duration, then
`mode line (from dep -> to arr)` per leg, walk legs included). This is the raw material; the
fuzzy match in the next cell is the verdict.

In [15]:
fun respOf(planner: String, req: String): TravelPlannerResponse? =
    collected.firstOrNull { it.planner == planner && it.result.requestId == req }?.result?.asResponse()

val full = StringBuilder()
for (req in collected.map { it.result.requestId }.distinct()) {
    val otpR = respOf("otp", req); val motisR = respOf("motis", req)
    if (otpR == null || motisR == null) continue
    val otp = otpItineraries(otpR); val motis = motisItineraries2(motisR)
    full.append("<h4>$req</h4>")
    full.append("<b>OTP — ${otp.size} itineraries</b><pre>")
    otp.forEachIndexed { i, it -> full.append("%2d. %s\n".format(i + 1, it.render())) }
    full.append("</pre><b>MOTIS — ${motis.size} itineraries</b><pre>")
    motis.forEachIndexed { i, it -> full.append("%2d. %s\n".format(i + 1, it.render())) }
    full.append("</pre>")
}
HTML(full.toString())

oslos-nationalth OTP — 23 itineraries 1. (16m) walk 16:09-16:25
 2. (10m) walk 16:09-16:11 -> BUS 37 (Jernbanetorget 16:11 -> Nationaltheatret 16:18) -> walk 16:18-16:19
 3. (11m) walk 16:11-16:16 -> BUS 81 (Jernbanetorget 16:16 -> Nationaltheatret 16:21) -> walk 16:21-16:22
 4. (9m) walk 16:15-16:17 -> TRAM 15 (Jernbanetorget 16:17 -> Nationaltheatret 16:23) -> walk 16:23-16:24
 5. (12m) walk 16:16-16:21 -> BUS 30 (Jernbanetorget 16:21 -> Nationaltheatret 16:27) -> walk 16:27-16:28
 6. (10m) walk 16:19-16:21 -> BUS 37 (Jernbanetorget 16:21 -> Nationaltheatret 16:28) -> walk 16:28-16:29
 7. (10m) walk 16:20-16:22 -> BUS 31 (Jernbanetorget 16:22 -> Nationaltheatret 16:29) -> walk 16:29-16:30
 8. (11m) walk 16:21-16:26 -> BUS 81 (Jernbanetorget 16:26 -> Nationaltheatret 16:31) -> walk 16:31-16:32
 9. (9m) walk 16:25-16:27 -> TRAM 15 (Jernbanetorget 16:27 -> Nationaltheatret 16:33) -> walk 16:33-16:34
10. (10m) walk 16:26-16:28 -> BUS 31 (Jernbanetorget 16:28 -> Nationaltheatret 16:35) -> walk 16:35-16:36
11. (10m) walk 16:29-16:31 -> BUS 37 (Jernbanetorget 16:31 -> Nationaltheatret 16:38) -> walk 16:38-16:39
12. (10m) walk 16:32-16:34 -> BUS 31 (Jernbanetorget 16:34 -> Nationaltheatret 16:41) -> walk 16:41-16:42
13. (9m) walk 16:35-16:37 -> TRAM 15 (Jernbanetorget 16:37 -> Nationaltheatret 16:43) -> walk 16:43-16:44
14. (10m) walk 16:38-16:40 -> BUS 31 (Jernbanetorget 16:40 -> Nationaltheatret 16:47) -> walk 16:47-16:48
15. (10m) walk 16:39-16:41 -> BUS 37 (Jernbanetorget 16:41 -> Nationaltheatret 16:48) -> walk 16:48-16:49
16. (11m) walk 16:41-16:46 -> BUS 81 (Jernbanetorget 16:46 -> Nationaltheatret 16:51) -> walk 16:51-16:52
17. (9m) walk 16:45-16:47 -> TRAM 15 (Jernbanetorget 16:47 -> Nationaltheatret 16:53) -> walk 16:53-16:54
18. (12m) walk 16:46-16:51 -> BUS 30 (Jernbanetorget 16:51 -> Nationaltheatret 16:57) -> walk 16:57-16:58
19. (10m) walk 16:49-16:51 -> BUS 37 (Jernbanetorget 16:51 -> Nationaltheatret 16:58) -> walk 16:58-16:59
20. (10m) walk 16:50-16:52 -> BUS 31 (Jernbanetorget 16:52 -> Nationaltheatret 16:59) -> walk 16:59-17:00
21. (11m) walk 16:51-16:56 -> BUS 81 (Jernbanetorget 16:56 -> Nationaltheatret 17:01) -> walk 17:01-17:02
22. (9m) walk 16:55-16:57 -> TRAM 15 (Jernbanetorget 16:57 -> Nationaltheatret 17:03) -> walk 17:03-17:04
23. (10m) walk 16:56-16:58 -> BUS 31 (Jernbanetorget 16:58 -> Nationaltheatret 17:05) -> walk 17:05-17:06
 MOTIS — 7 itineraries 1. (5m) walk 16:09-16:10 -> METRO 2 (Jernbanetorget 16:10 -> Nationaltheatret 16:13) -> walk 16:13-16:14
 2. (5m) walk 16:12-16:13 -> METRO 3 (Jernbanetorget 16:13 -> Nationaltheatret 16:16) -> walk 16:16-16:17
 3. (5m) walk 16:14-16:15 -> METRO 4 (Jernbanetorget 16:15 -> Nationaltheatret 16:18) -> walk 16:18-16:19
 4. (5m) walk 16:17-16:18 -> METRO 2 (Jernbanetorget 16:18 -> Nationaltheatret 16:21) -> walk 16:21-16:22
 5. (5m) walk 16:19-16:20 -> METRO 5 (Jernbanetorget 16:20 -> Nationaltheatret 16:23) -> walk 16:23-16:24
 6. (5m) walk 16:21-16:22 -> METRO 1 (Jernbanetorget 16:22 -> Nationaltheatret 16:25) -> walk 16:25-16:26
 7. (5m) walk 16:23-16:24 -> METRO 5 (Jernbanetorget 16:24 -> Nationaltheatret 16:27) -> walk 16:27-16:28
 oslos-majorstuen OTP — 44 itineraries 1. (42m) walk 16:09-16:51
 2. (20m) walk 16:11-16:18 -> METRO 2 (Jernbanetorget 16:18 -> Majorstuen 16:24) -> walk 16:24-16:31
 3. (19m) walk 16:12-16:14 -> TRAM 19 (Jernbanetorget 16:14 -> Majorstuen 16:29) -> walk 16:29-16:31
 4. (20m) walk 16:15-16:22 -> METRO 1 (Jernbanetorget 16:22 -> Majorstuen 16:28) -> walk 16:28-16:35
 5. (20m) walk 16:15-16:17 -> TRAM 15 (Jernbanetorget 16:17 -> Frogner stadion 16:32) -> walk 16:32-16:36
 6. (20m) walk 16:17-16:24 -> METRO 5 (Jernbanetorget 16:24 -> Majorstuen 16:30) -> walk 16:30-16:37
 7. (20m) walk 16:18-16:25 -> METRO 2 (Jernbanetorget 16:25 -> Majorstuen 16:31) -> walk 16:31-16:38
 8. (20m) walk 16:21-16:28 -> METRO 3 (Jernbanetorget 16:28 -> Majorstuen 16:34) -> walk 16:34-16:41
 9. (19m) walk 16:22-16:24 -> TRAM 19 (Jernbanetorget 16:24 -> 

## 12. Fuzzy semantic match (transit legs)

Per request, the transit itineraries each engine found, matched on line + minute-precision dep/arr
(walk legs ignored): `=` found by **both**, `OTP` only, or `MOTIS` only. `itinerariesMatchOtp` is 1.0
exactly when there is nothing in the OTP-only or MOTIS-only rows.

_A walk-only itinerary has no transit legs, so it never appears here — that's why OTP's direct-walk
option doesn't show as a spurious difference._

In [16]:
val diff = StringBuilder()
for (req in collected.map { it.result.requestId }.distinct()) {
    val otpR = respOf("otp", req); val motisR = respOf("motis", req)
    if (otpR == null || motisR == null) continue
    val otp = transitFingerprints(otpItineraries(otpR))
    val motis = transitFingerprints(motisItineraries2(motisR))
    val matched = otp == motis
    diff.append("<h4>$req &mdash; ${if (matched) "match" else "differ"}</h4>")
    diff.append("<table><tr><th>in</th><th>transit journey (line dep-arr)</th></tr>")
    for (fp in (otp + motis).sortedBy { it.joinToString(" ") }) {
        val mark = if (fp in otp && fp in motis) "=" else if (fp in otp) "OTP" else "MOTIS"
        diff.append("<tr><td>$mark</td><td>${fp.joinToString(", ")}</td></tr>")
    }
    diff.append("</table>")
}
HTML(diff.toString())

in,transit journey (line dep-arr)
OTP,BUS 30 16:21-16:27
OTP,BUS 30 16:51-16:57
OTP,BUS 31 16:22-16:29
OTP,BUS 31 16:28-16:35
OTP,BUS 31 16:34-16:41
OTP,BUS 31 16:40-16:47
OTP,BUS 31 16:52-16:59
OTP,BUS 31 16:58-17:05
OTP,BUS 37 16:11-16:18
OTP,BUS 37 16:21-16:28


## 13. Stop both

In [ ]:
otp.stop(otpV)
motis.stop(motisV)